In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    cross_validate
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

df = pd.read_csv("data/credito.csv")

df.head()

X = df.drop(
    columns=[
        "id_contrato",
        "inadimplente"
    ]
)

y = df["inadimplente"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_treino.shape)
print(X_teste.shape)

numericas = X.select_dtypes(
    include=np.number
).columns.tolist()

categoricas = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numéricas:")
print(numericas)

print("\nCategóricas:")
print(categoricas)

(4800, 9)
(1200, 9)
Numéricas:
['idade', 'renda_mensal', 'tempo_emprego_anos', 'score_credito', 'dividas_ativas', 'prazo_meses', 'valor_emprestimo']

Categóricas:
['possui_imovel', 'finalidade']


In [2]:
pipeline_num = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

pipeline_cat = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

preprocessador = ColumnTransformer([
    (
        "numericas",
        pipeline_num,
        numericas
    ),
    (
        "categoricas",
        pipeline_cat,
        categoricas
    )
])

In [3]:
modelos = {

    "Regressão Logística":
        LogisticRegression(
            max_iter=1000
        ),

    "Random Forest":
        RandomForestClassifier(
            random_state=42
        ),

    "Gradient Boosting":
        HistGradientBoostingClassifier(
            random_state=42
        )
}

In [4]:
resultados = []

for nome, modelo in modelos.items():

    pipeline = Pipeline([
        (
            "preprocessamento",
            preprocessador
        ),
        (
            "modelo",
            modelo
        )
    ])

    cv = cross_validate(
        pipeline,
        X_treino,
        y_treino,
        cv=5,
        scoring="roc_auc"
    )

    resultados.append({

        "Modelo": nome,

        "ROC_AUC_Média":
            cv["test_score"].mean(),

        "ROC_AUC_Desvio":
            cv["test_score"].std()
    })

In [5]:
comparacao = pd.DataFrame(
    resultados
).sort_values(
    "ROC_AUC_Média",
    ascending=False
)

comparacao

,Modelo,ROC_AUC_Média,ROC_AUC_Desvio
0,Regressão Logística,0.778072,0.022435
2,Gradient Boosting,0.747381,0.026125
1,Random Forest,0.741777,0.022977


In [6]:
melhor_modelo = comparacao.iloc[0]

print(
    "Melhor modelo:"
)

print(
    melhor_modelo["Modelo"]
)

Melhor modelo:
Regressão Logística


## Conclusão

Os dados foram divididos em conjuntos de treino e teste utilizando amostragem estratificada, preservando a proporção das classes. Foi construído um Pipeline contendo imputação de valores ausentes, padronização de variáveis numéricas e codificação das variáveis categóricas, evitando vazamento de dados. Três algoritmos foram comparados por validação cruzada de 5 folds utilizando ROC AUC. A Regressão Logística apresentou o melhor desempenho, com ROC AUC médio de 0,7781 ± 0,0224, superando os modelos Random Forest e Gradient Boosting. Dessa forma, a Regressão Logística foi selecionada para a etapa de avaliação.
